# 004 Run Deep Agent

这是 Deep Agents 学习线的第四份 Notebook，也是 quickstart 系列的最后一课。

上一课产物：

- 一个配置好 model 和 tools 的 Deep Agent
- 理解了 `create_deep_agent` 的参数

本课产物：

- 流式运行 agent 并实时观察执行过程
- 理解 agent 的规划、工具调用、结果综合流程

配套官方文档：

- [Quickstart - Run Agent](https://docs.langchain.com/oss/python/deepagents/quickstart)

学习目标：

1. 使用 `stream` 方法流式运行 agent，实时观察执行过程。
2. 解析并打印 agent 的 todo list 规划事件。
3. 解析并打印 agent 的工具调用事件。
4. 解析并打印 agent 的最终输出。
5. 理解 agent 的执行流程。

## 0. 加载项目配置

统一从 `.env` 读取配置。

In [ ]:
import os
from pathlib import Path
from pprint import pprint

from dotenv import load_dotenv

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / 'requirements.txt').exists() and (path / 'notebooks').exists():
            return path
    return current

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / '.env', override=False)

MODEL_CONFIG = {
    'api_key': os.getenv('OPENAI_API_KEY', 'EMPTY'),
    'model': os.getenv('OPENAI_MODEL', 'qwq'),
    'base_url': os.getenv('OPENAI_BASE_URL', 'http://192.168.102.19:8082/v1'),
}

safe_config = dict(MODEL_CONFIG)
safe_config['api_key'] = '***'
pprint(safe_config)

## 1. 创建 Agent

复用前三课的配置。

In [ ]:
from tavily import TavilyClient
from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

# 创建模型
model = ChatOpenAI(
    model=MODEL_CONFIG['model'],
    api_key=MODEL_CONFIG['api_key'],
    base_url=MODEL_CONFIG['base_url'],
)

# 创建搜索工具
tavily_api_key = os.getenv('TAVILY_API_KEY', '')

def internet_search(query: str) -> str:
    """Search the internet for information about the given query.

    Use this tool when you need to find information on the internet.
    Returns a formatted string with search results.

    Args:
        query: The search query string.

    Returns:
        Formatted search results as a string.
    """
    client = TavilyClient(api_key=tavily_api_key)
    results = client.search(query=query, max_results=5)

    formatted = []
    for i, result in enumerate(results.get('results', []), 1):
        title = result.get('title', 'N/A')
        url = result.get('url', 'N/A')
        content = result.get('content', 'N/A')
        formatted.append(f'[{i}] {title}\nURL: {url}\n{content}\n')

    return '\n'.join(formatted) if formatted else 'No results found.'

# 创建 agent
agent = create_deep_agent(
    model=model,
    tools=[internet_search],
    system_prompt='你是一个研究助手，擅长收集信息并撰写报告。',
)

print('Agent 创建成功')

## 2. 流式运行 Agent

使用 `stream` 方法流式运行 agent，实时打印执行事件。

这样可以直观看到 agent 在做什么：

```text
[规划] Agent 正在拆分任务...
  - 搜索 LangGraph 基本概念
  - 整理核心特性
  - 撰写介绍报告
[工具调用] internet_search: LangGraph 是什么
[工具结果] 返回 5 条搜索结果
[回答] Agent 正在撰写最终回答...
```

注意：这会调用 Tavily API 和大模型，需要一定时间。

下面先打印所有原始 event 结构，帮助理解 `write_todos` 的格式。

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage, HumanMessage, SystemMessage

if not tavily_api_key:
    print('跳过运行测试：TAVILY_API_KEY 未配置')
    print('配置后可运行此单元格')
else:
    print('=== 逐行打印模型返回的所有内容 ===')
    print()

    event_count = 0
    for event in agent.stream({
        'messages': [{'role': 'user', 'content': '根据互联网最新的大学生就业数据，写一份关于大学生就业的研究报告'}],
    }):
        event_count += 1
        print(f'--- Event #{event_count} ---')
        for key, value in event.items():
            print(f'  Key: {key}')
            if value is None:
                print('    Value: None')
            elif isinstance(value, dict):
                for k2, v2 in value.items():
                    if k2 == 'messages' and isinstance(v2, list):
                        print(f'    {k2}: (list of {len(v2)} messages)')
                        for i, msg in enumerate(v2):
                            print(f'      [{i}] {msg.__class__.__name__}')
                            if isinstance(msg, HumanMessage):
                                print(f'        role: user')
                                print(f'        content: {msg.content}')
                            elif isinstance(msg, SystemMessage):
                                print(f'        role: system')
                                print(f'        content: {msg.content[:200]}...')
                            elif isinstance(msg, AIMessage):
                                print(f'        role: assistant')
                                if msg.content:
                                    print(f'        content: {msg.content}')
                                if msg.tool_calls:
                                    print(f'        tool_calls:')
                                    for tc in msg.tool_calls:
                                        print(f'          - name: {tc.get("name")}')
                                        print(f'            args: {tc.get("args")}')
                                        print(f'            id: {tc.get("id")}')
                                if msg.response_metadata:
                                    print(f'        response_metadata: {msg.response_metadata}')
                            elif isinstance(msg, ToolMessage):
                                print(f'        role: tool')
                                print(f'        name: {msg.name}')
                                print(f'        tool_call_id: {msg.tool_call_id}')
                                print(f'        content: {msg.content}')
                            else:
                                print(f'        content: {msg}')
                    else:
                        print(f'    {k2}: {v2}')
            else:
                print(f'  Value: {value}')
        print()

## 3. 观察执行过程

Deep Agent 会自动执行以下步骤：

```text
1. 规划：拆成多个子任务（write_todos）
2. 研究：调用 internet_search 收集信息
3. 存储：把搜索结果写入文件（管理大上下文）
4. 综合：基于搜索结果写报告
```

上面的流式输出已经实时打印了每个步骤。

你也可以保存完整结果，后续查看中间状态。

In [ ]:
if not tavily_api_key:
    print('跳过检查：TAVILY_API_KEY 未配置')
else:
    print('流式输出已在上一个单元格展示')
    print()
    print('=== 如何获取 agent 生成的报告 ===')
    print()
    print('注意：deep agents 使用虚拟文件系统（内存中），不是真实磁盘文件。')
    print('agent 说 "报告已保存" 只是写入了内存中的虚拟文件。')
    print()
    print('如果需要真实保存到磁盘，需要自定义 write_file 工具。')
    print('或者从 result 的 ToolMessage 中提取 write_file 的返回内容。')

## 4. Quickstart 系列小结

这四课完成了：

| 课时 | 内容 | 产物 |
|---|---|---|
| 001 | Deep Agents 概览 | 心智模型、模型网关验证 |
| 002 | 创建搜索工具 | `internet_search` 工具 |
| 003 | 创建 Deep Agent | 配置好的 agent 实例 |
| 004 | 运行 Agent | 流式输出执行过程 |

你现在可以：

1. 理解 Deep Agents 的核心能力
2. 创建和注册工具
3. 使用 `create_deep_agent` 创建 agent
4. 流式运行 agent 并实时观察执行过程

## 后续学习方向

1. **Customization**：自定义 agent 行为
2. **Harness**：更底层的控制
3. **多工具集成**：添加文件读写、代码执行等工具
4. **生产部署**：LangSmith 集成与监控

## 5. 练习

请你思考后回答：

1. Agent 执行一个复杂任务时，会自动做哪些事情？
2. 如果 agent 没有调用搜索工具，可能是什么原因？
3. 你想给这个 agent 添加什么新工具？为什么？